# BF3 — Trajectory Profiles

Extracts 28-dimensional semantic feature vectors from museum trajectories
and computes pairwise similarity. Uses `semantic_features.py` and `semantic_similarity.py`.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('helpers'))
from semantic_features import (
    SemanticFeatureExtractor, normalize_features,
    FEATURE_GROUPS, TOTAL_DIMENSIONS
)
from semantic_similarity import TrajectorySimilarityEngine

## 1. Extract Features

In [ ]:
extractor = SemanticFeatureExtractor()
print(f'Feature dimensions: {TOTAL_DIMENSIONS}')
print(f'Feature groups: {FEATURE_GROUPS}')
print(f'Feature names ({len(extractor.get_feature_names())}):')
for name in extractor.get_feature_names():
    print(f'  {name}')

In [ ]:
# Extract features for all trajectories
features = extractor.batch_extract_features()
print(f'\nExtracted features for {len(features)} trajectories')

# Save
from pathlib import Path
output_dir = Path('output/semantic_features')
extractor.save_features(features, output_dir / 'trajectory_features.json')
extractor.save_metadata(output_dir / 'feature_metadata.json')

## 2. Feature Analysis

In [ ]:
# Convert to matrix for analysis
uris = list(features.keys())
X = np.array([features[uri] for uri in uris])
feature_names = extractor.get_feature_names()

# Feature statistics
print('Feature statistics:')
for group_name, dim in FEATURE_GROUPS.items():
    groups = extractor.get_feature_groups()
    start, end = groups[group_name]
    group_data = X[:, start:end]
    print(f'\n  {group_name} (dims {start}-{end}):')
    print(f'    Mean: {group_data.mean():.4f}')
    print(f'    Std:  {group_data.std():.4f}')
    print(f'    Min:  {group_data.min():.4f}')
    print(f'    Max:  {group_data.max():.4f}')

In [ ]:
# Visualize feature distributions per group
groups = extractor.get_feature_groups()
fig, axes = plt.subplots(1, len(groups), figsize=(20, 4))

for ax, (group_name, (start, end)) in zip(axes, groups.items()):
    group_data = X[:, start:end]
    ax.boxplot(group_data, labels=[f'{i}' for i in range(end-start)])
    ax.set_title(f'{group_name} ({end-start}d)')
    ax.tick_params(axis='x', labelsize=6)

plt.tight_layout()
plt.show()

## 3. Similarity Analysis

In [ ]:
# Normalize features
norm_features = normalize_features(features, groups)

# Compute similarity engine
engine = TrajectorySimilarityEngine(semantic_features=norm_features)
engine._semantic_uris = list(norm_features.keys())
engine._semantic_matrix = np.array([norm_features[uri] for uri in engine._semantic_uris])

# Compute similarity matrix
sim_matrix, sim_uris = engine.compute_semantic_similarity_matrix()
print(f'Similarity matrix shape: {sim_matrix.shape}')
print(f'Mean similarity: {sim_matrix[np.triu_indices_from(sim_matrix, k=1)].mean():.4f}')
print(f'Std similarity:  {sim_matrix[np.triu_indices_from(sim_matrix, k=1)].std():.4f}')

In [ ]:
# Similarity heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='viridis', aspect='auto')
ax.set_title('Trajectory Semantic Similarity Matrix')
plt.colorbar(im, ax=ax, label='Cosine Similarity')
plt.tight_layout()
plt.show()

In [ ]:
# Find most similar pairs
if len(sim_uris) > 1:
    query_uri = sim_uris[0]
    similar = engine.find_similar_semantic(query_uri, top_k=5)
    q_short = query_uri.split('#')[-1] if '#' in query_uri else query_uri.split('/')[-1]
    print(f'Most similar to {q_short}:')
    for uri, sim in similar:
        short = uri.split('#')[-1] if '#' in uri else uri.split('/')[-1]
        print(f'  {short}: {sim:.4f}')

    # Breakdown
    if similar:
        top_match = similar[0][0]
        breakdown = engine.get_similarity_breakdown(query_uri, top_match)
        print('\nSimilarity breakdown with top match:')
        for group, score in breakdown.items():
            print(f'  {group}: {score:.4f}')

In [ ]:
# Save similarity data
engine.save_similarity_data('output/similarity')